> **作成の記録**：このノートブックは本人の依頼で Claude が試しに作り、本人の指示なしに実行した（2026-11-05）。学校用には後で本人が再現する。各セル先頭に書いた人を明記。

# 05 v1.1：物差しを直して「前」の数字を出し直す

**やること**：検索はそのまま、測り方だけ直す。ここの数字が v2.0 と比べる「前」になる。
1. `has_trap` の修正（"without X"・"X-free" を罠にしない）
2. E か E 以外かの2択 nDCG
3. 本文（説明文＋箇条書き）の除外語も数える（タイトルには見えない罠）

関数は `src/v11_metrics.py`（Claude が作成・本人依頼）。データ・検索の関数は 03 と同じ。

⚠️ 本文の判定も単語を探すだけで、意味は読んでいない。「本文にある＝必ず罠」ではないので、v2.0 の正解ラベルの候補として見る。

In [1]:
# ✍️ Claude が作成（読み込みの行は 03 のセル1・3 と同じ内容）
import re, sys
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

sys.path.append('..')
from src.v11_metrics import add_trap_columns, rate_at_k, trap_floor, ndcg_binary_at_k

df_ex = pd.read_parquet('../data/examples_us.parquet')
df_pr = pd.read_parquet('../data/products_us.parquet')
df_merged = pd.merge(df_ex, df_pr, on=['product_id', 'product_locale'], how='left')
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/yyyyy/Mahoroba/constraintrag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1105 11:21:49.997000 84893 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8293.75it/s]

> ✍️ 見出し・説明は Claude
## 03 の関数（変更なし）

In [2]:
# 👤 本人のコードを Claude がコピー（内容は変更なし。確認用の行だけ除いた）
# 出どころ：notebooks/03_eval.ipynb のセル 5 get_candidates／7 tokenize・rank_tfidf・rank_bm25／34 rank_embed／9 ndcg_at_k・trap_rate_at_k
def get_candidates(query_id, trap_word):
    # 1. このクエリの候補だけにする
    df_q = df_merged[df_merged['query_id'] == query_id].copy()

    # 2. doc 列を作る（タイトル＋説明 → HTML タグを消す）
    df_q['doc'] = df_q['product_title'].fillna('') + ' ' + df_q['product_description'].fillna('')
    df_q['doc'] = df_q['doc'].str.replace(r'<[^>]+>', ' ', regex=True)

    # 3. has_trap 列を作る（タイトルに trap_word があるか）
    df_q['has_trap'] = df_q['product_title'].str.contains(trap_word, case=False, regex=False).astype(int)

    return df_q  # 候補の表（DataFrame）


def tokenize(text):
    # 小文字にして、2文字以上の単語を取り出す
    text = text.lower()
    words = re.findall(r'\b\w\w+\b', text)
    return words  # 単語のリスト


def rank_tfidf(df_q, query):
    # 1. TfidfVectorizer で doc 列をベクトルにする（fit_transform）
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df_q['doc'])
    

    # 2. クエリを同じ辞書でベクトルにする（transform）
    query_vector = vectorizer.transform([query])

    # 3. コサインで点数を出す
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    # 4. score 列を足して、点の高い順に並べる
    df_q['score'] = scores
    df_q_sorted = df_q.sort_values(by='score', ascending=False)
    return df_q_sorted  # 並べた表


def rank_bm25(df_q, query):
    # 1. doc 列を全部 tokenize して BM25Okapi に渡す
    tokenized_docs = df_q['doc'].apply(tokenize).tolist()
    bm25 = BM25Okapi(tokenized_docs)
    

    # 2. クエリも tokenize して点数を出す（get_scores）
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    # 3. score 列を足して、点の高い順に並べる
    df_q['score'] = scores
    df_q_sorted = df_q.sort_values(by='score', ascending=False)
    return df_q_sorted  # 並べた表


def rank_embed(df_q, query):
    # 1. doc 列を model.encode でベクトルにする（候補数 × 384）→ doc_vectors
    doc_vectors = model.encode(df_q['doc'].tolist())
    # 2. クエリも model.encode（1 × 384）※ [query] とリストで渡す → query_vector
    query_vector = model.encode([query])
    # 3. cosine_similarity で点数 → .flatten()（↓ rank_tfidf の 3 をそのまま。変数名だけ 1・2 で作ったものに合わせる）
    scores = cosine_similarity(query_vector, doc_vectors).flatten()

    # 4. score 列を足して、高い順に並べて返す（↓ rank_tfidf の 4 をそのまま）
    df_q['score'] = scores
    df_q_sorted = df_q.sort_values(by='score', ascending=False)
    return df_q_sorted  # 並べた表


GAIN = {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0}


def ndcg_at_k(ranked, k=10):
    # 1. 正解の点：esci_label を GAIN で数字にする
    ranked['esci_gain'] = ranked['esci_label'].map(GAIN)

    # 2. モデルの点：score 列
    
    ranked['score'] = ranked['score'].fillna(0)

    # 3. ndcg_score に渡す（両方 [ ] で包む、k も渡す）
    ndcg = ndcg_score([ranked['esci_gain'].values], [ranked['score'].values], k=k)
    return  ndcg  # 0〜1 の数字


def trap_rate_at_k(ranked, k=10):
    # 上位 k 件の has_trap の数 ÷ 枠の数（候補が k 件未満なら候補の数）
    slots = min(k, len(ranked))
    trap_rate = ranked.head(k)['has_trap'].sum() / slots
    return  trap_rate  # 0〜1 の数字

In [3]:
# 👤 本人のクエリ一覧を Claude がコピー（変更なし）。出どころ：03_eval.ipynb のセル 17
# ✅ 簡単な20件（左の番号＝シャッフル後の順番）。上の7件の queries を置き換える
queries = [
    (71987, 'kn95'),         # 30  n95 not kn95 masks
    (23909, 'chicken'),      # 37  cat treats without chicken
    (115782, 'hood'),        # 71  zippered sweatshirt without hood mens
    (1435, 'controller'),    # 79  100 watt folding solar panel without charge controller
    (59940, 'chromebook'),   # 90  laptop not chromebook
    (40132, 'pop it'),       # 93  fidget pack without pop it
    (1509, 'controller'),    # 107 100w solar panel without controller
    (1400, 'microphone'),    # 124 100 headset without micrphone（クエリは打ち間違い）
    (78323, 'microphone'),   # 148 pc headset without microphone
    (99719, 'hood'),         # 152 sweatshirt without hood for men
    (28603, 'straw'),        # 156 contigo without straw
    (7752, 'snooze'),        # 166 alarm clock without snooze
    (42300, 'cartridge'),    # 180 fountain pen without cartridge
    (14476, 'cup holder'),   # 197 bed desk without cup holder
    (95852, 'motion'),       # 219 solar lights not motion sensor
    (49760, 'wireless'),     # 271 headphones not wireless
    (45968, 'ottoman'),      # 274 glider without ottoman
    (77730, 'basket'),       # 307 paper shredder without basket
    (75456, 'extra virgin'), # 320 olive oil not extra virgin
    (447, 'gel'),            # 325 0.5 pens fine point black not gel
]

> ✍️ 見出しは Claude
## 1. 20クエリを測り直す

In [4]:
# ✍️ Claude が作成。集計の形は 03 のセル 36（本人）を参考にした
rows = []
for qid, trap in queries:
    df_q = add_trap_columns(get_candidates(qid, trap), trap)
    query_text = df_q['query'].iloc[0]
    floor = trap_floor(df_q, 'has_trap')
    for name, rank in [('TF-IDF', rank_tfidf), ('BM25', rank_bm25), ('埋め込み', rank_embed)]:
        ranked = rank(df_q.copy(), query_text)
        rows.append({
            'query_id': qid, 'query': query_text, 'trap_word': trap, 'model': name,
            'ndcg@10': ndcg_at_k(ranked), 'ndcg_binary@10': ndcg_binary_at_k(ranked),
            'trap_old@10': rate_at_k(ranked, 'has_trap_old'), 'trap@10': rate_at_k(ranked, 'has_trap'),
            'trap_floor': floor, 'hidden@10': rate_at_k(ranked, 'hidden_trap'),
            'body@10': rate_at_k(ranked, 'has_body'),
        })
results_v11 = pd.DataFrame(rows)
results_v11['excess_trap'] = results_v11['trap@10'] - results_v11['trap_floor']
results_v11.groupby('model').mean(numeric_only=True).drop(columns='query_id').round(3)

,ndcg@10,ndcg_binary@10,trap_old@10,trap@10,trap_floor,hidden@10,body@10,excess_trap
model,,,,,,,,
BM25,0.389,0.358,0.755,0.730,0.2,0.125,0.975,0.530
TF-IDF,0.380,0.345,0.765,0.745,0.2,0.095,0.965,0.545
埋め込み,0.373,0.338,0.790,0.780,0.2,0.100,0.985,0.580


> ✍️ 読み方は Claude が作成
## 2. 読み方
- `trap_old@10` が 03 の 0.755／0.765／0.79 と同じなら、検索は変わっていない
- `trap_old@10` → `trap@10` の差：判定のズレで悪く見えていた分
- `hidden@10`：タイトルは問題ないが、本文に除外語がある商品の割合（タイトルだけでは測れていなかった分）
- `body@10`：上位10件のうち本文がある割合。低いと Agent が読むものがない

In [5]:
# ✍️ Claude が作成
# has_trap が変わった商品を目で確かめる（判定の確認）
for qid, trap in queries:
    d = add_trap_columns(get_candidates(qid, trap), trap)
    diff = d[d['has_trap_old'] != d['has_trap']]
    if len(diff):
        print(f"[{qid}] {trap}: {len(diff)}件")
        for t in diff['product_title'].head(3):
            print('   ', t[:90])

[1509] controller: 1件
    Renogy 100W Eclipse Lightweight Suitcase Without Controller, Panel Only, Black
[77730] basket: 4件
    Aurora AU800SD Professional Strip Cut Paper Shredder/CD/Credit Card Shredder Without Waste
    Aurora AS680S Professional Light Duty Strip Cut Paper Shredder Without Wastebasket
    Aurora AU800SD Professional Strip Cut Paper Shredder/CD/Credit Card Shredder Without Waste


In [6]:
# ✍️ Claude が作成
# クエリごと（BM25）
results_v11[results_v11['model'] == 'BM25'][['query', 'trap_word', 'ndcg_binary@10', 'trap_old@10', 'trap@10', 'trap_floor', 'hidden@10', 'body@10']].round(2)

,query,trap_word,ndcg_binary@10,trap_old@10,trap@10,trap_floor,hidden@10,body@10
1,n95 not kn95 masks,kn95,0.28,0.8,0.8,0.4,0.0,0.9
4,cat treats without chicken,chicken,0.23,1.0,1.0,0.0,0.0,1.0
7,zippered sweatshirt without hood mens,hood,0.52,0.7,0.7,0.0,0.0,1.0
10,100 watt folding solar panel without charge co...,controller,0.40,0.6,0.6,0.0,0.0,1.0
13,laptop not chromebook,chromebook,0.00,1.0,1.0,0.6,0.0,1.0
16,fidget pack without pop it,pop it,0.25,0.1,0.1,0.0,0.1,0.9
19,100w solar panel without controller,controller,0.69,0.7,0.6,0.2,0.1,1.0
22,100 headset without micrphone,microphone,0.15,0.8,0.8,0.4,0.2,1.0
25,pc headset without microphone,microphone,0.60,0.4,0.4,0.1,0.6,1.0
28,sweatshirt without hood for men,hood,0.44,0.7,0.7,0.0,0.0,1.0


> 👤 ここは本人が書く欄
## 3. 気づき（自分の言葉で）


In [ ]:
.p